# Auris: train and test the ear-image model

This notebook trains the image model for the Auris Android app, then tells you honestly how accurate it is.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell in order with ▶.

What it does:
1. Collects your labelled ear images from Google Drive and removes exact duplicates.
2. Keeps near-identical photos together (and photos of the same patient, if you give a list), so the test is fair.
3. Trains an EfficientNet model that starts from ImageNet weights.
4. Tests it on images it has never seen: accuracy with a likely range, sensitivity for each condition, and a confusion matrix.
5. Picks a *confident mode* threshold that aims for at least 95% accuracy on the cases Auris answers by itself.
6. Exports `auris.onnx`, `labels.txt` and `model_info.json` for the app.

In [ ]:
#@title 1. Set up (about a minute)
!pip -q install onnx onnxscript onnxruntime

import os, re, json, math, random, shutil, hashlib, zipfile, datetime
import numpy as np
import pandas as pd
import torch, torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| device:', DEVICE)
if DEVICE == 'cpu':
    print('⚠ No GPU. Use Runtime > Change runtime type > T4 GPU, otherwise training will be very slow.')

## Put your images in Google Drive

Make a folder called `auris_raw` in My Drive. Inside it, put one folder per condition (subfolders are fine). Folder names are matched automatically, for example `Normal`, `Earwax`, `AOM`, `Otitis media with effusion`, `Chronic otitis media`, `Perforation`, `Otitis externa`, `Otomycosis`, `Tympanosclerosis`, `Myringosclerosis`, `Retraction`, `Cholesteatoma`, `Ventilation tube`, `Foreign body`, `Exostosis` and `Poor quality`. Folders that don't match are listed and skipped: rename them, or add their names to `ALIASES` in step 3.

Label quality decides accuracy. Use images whose diagnosis an ENT has confirmed, ideally agreed by two ENTs.

**Public datasets to start with** (check each licence before any commercial use):
- **Eardrum dataset**, Özel Van Akdamar Hospital, Turkey: 956 images, each reviewed by three ENT specialists. https://doi.org/10.6084/m9.figshare.13648166 (also on Kaggle as `erdalbasaran/eardrum-dataset-otitis-media`)
- **Ear Imagery dataset**, Universidad de Chile: 880 images from 180 patients (normal, earwax plug, myringosclerosis, chronic otitis media). https://doi.org/10.6084/m9.figshare.11886630. Many of its images are near-duplicates; step 4 keeps them on the same side of the split.
- **Otoscopic image dataset for ear diseases**, IEEE DataPort, India: 600 hospital images in 14 classes, verified by an ENT specialist. https://ieee-dataport.org/documents/otoscopic-image-dataset-ear-diseases (may need an IEEE DataPort account)

Public data alone is small and uneven. Reaching 95% across many conditions will need images from your own partner clinics, collected with patient consent and with names removed.

**Optional:** if you know which images came from the same patient, make a CSV with the columns `file` (the image file name) and `patient_id`, upload it to Drive, and set `PATIENT_CSV` below.

In [ ]:
#@title 2. Settings
from google.colab import drive
drive.mount('/content/drive')

RAW_DIRS = ['/content/drive/MyDrive/auris_raw']  # add more folders to merge several datasets
PATIENT_CSV = ''          # optional: path to a CSV with columns file,patient_id
OUT_DIR = '/content/drive/MyDrive/auris_model'   # results are saved here
ARCH = 'efficientnet_b0'  # 'efficientnet_b0' (small, fast) | 'efficientnet_b2' | 'convnext_tiny' (bigger, slower)
MIN_IMAGES = 30           # conditions with fewer images are left out of this model
EPOCHS = 30
BATCH = 32
TARGET = 0.95

os.makedirs(OUT_DIR, exist_ok=True)
for d in RAW_DIRS:
    print(('Found:   ' if os.path.isdir(d) else 'MISSING: ') + d)

## Organise the images
Every image is converted to RGB JPEG (longest side 1024 px), named by its content, and filed under an Auris condition id. Exact copies are kept once. An image filed under two different conditions is removed, because its label can't be trusted.

In [ ]:
#@title 3. Organise the images
# Folder names are matched to Auris condition ids. More specific names come first.
ALIASES = [
    ('poor_quality',     ['poor quality', 'unclear', 'ungradable', 'blurry']),
    ('otomycosis',       ['otomycosis', 'fungal', 'fungus']),
    ('oe',               ['otitis externa', 'external otitis']),
    ('cholesteatoma',    ['cholesteatoma', 'retraction']),
    ('tympanosclerosis', ['tympanosclerosis', 'tympanoskleros', 'myringosclerosis', 'miringoskleroz', 'calcification']),
    ('grommet',          ['grommet', 'ventilation tube', 'tympanostomy tube', 'ventilating tube']),
    ('foreign_body',     ['foreign']),
    ('bullous',          ['bullous']),
    ('haemotympanum',    ['haemotympanum', 'hemotympanum']),
    ('exostosis',        ['exostosis', 'exostoses', 'osteoma']),
    ('wax',              ['earwax', 'ear wax', 'cerumen', 'wax']),
    ('csom',             ['csom', 'chronic', 'chornic', 'perforation']),
    ('ome',              ['ome', 'effusion', 'serous', 'secretory', 'glue ear']),
    ('aom',              ['aom', 'acute otitis media', 'acute']),
    ('normal',           ['normal', 'healthy']),
]
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
DATA = '/content/auris_data'

def plural(n, word):
    return f'{n} {word}' + ('' if n == 1 else 's')

def _norm(s):
    return ' ' + re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip() + ' '

def class_for(rel_path):
    """Walks up from the image's own folder and returns the first condition id that matches."""
    for folder in reversed(rel_path.split(os.sep)[:-1]):
        name = _norm(folder)
        for cid, words in ALIASES:
            if any(' ' + w + ' ' in name for w in words):
                return cid
    return None

if os.path.exists(DATA):
    shutil.rmtree(DATA)
rows, skipped, owner, conflicts, unreadable = [], {}, {}, set(), 0
for root in RAW_DIRS:
    for dirpath, _, files in os.walk(root):
        for fn in sorted(files):
            if os.path.splitext(fn)[1].lower() not in EXTS:
                continue
            src = os.path.join(dirpath, fn)
            rel = os.path.relpath(src, root)
            cid = class_for(rel)
            if cid is None:
                key = os.path.dirname(rel) or '(top level)'
                skipped[key] = skipped.get(key, 0) + 1
                continue
            try:
                im = ImageOps.exif_transpose(Image.open(src)).convert('RGB')
            except Exception:
                unreadable += 1
                continue
            im.thumbnail((1024, 1024))
            digest = hashlib.sha1(im.tobytes()).hexdigest()[:16]
            if digest in owner:
                if owner[digest] != cid:
                    conflicts.add(digest)
                continue
            owner[digest] = cid
            dst = os.path.join(DATA, cid, digest + '.jpg')
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            im.save(dst, quality=95)
            rows.append({'path': dst, 'label': cid, 'source': rel, 'digest': digest})

assert rows, 'No usable images found. Check RAW_DIRS and the folder names.'
df = pd.DataFrame(rows)
if conflicts:
    for p in df[df.digest.isin(conflicts)].path:
        os.remove(p)
    df = df[~df.digest.isin(conflicts)]
    print(f'Removed {plural(len(conflicts), "image")} filed under two different conditions.')
if skipped:
    print('Skipped folders (no matching condition). Rename them or add them to ALIASES:')
    for k, v in sorted(skipped.items()):
        print(f'   {v:5d}  {k}')
if unreadable:
    print(f'{plural(unreadable, "file")} could not be opened and skipped.')

counts = df.label.value_counts()
too_few = counts[counts < MIN_IMAGES]
if len(too_few):
    print(f'Left out for now (fewer than {MIN_IMAGES} images):', dict(too_few))
df = df[~df.label.isin(too_few.index)].reset_index(drop=True)
print(f'\nUsing {len(df)} unique images in {df.label.nunique()} conditions:')
print(df.label.value_counts().to_string())

## Make the test fair
Frames from the same video, or several photos of one ear, look almost identical. If one lands in training and its twin in testing, the score looks better than it really is. This step links near-identical images (and same-patient images, if you gave a list) so each linked group stays entirely in train, validation or test. About 71% of images go to training, 14% to validation and 14% to the final test.

In [ ]:
#@title 4. Link near-duplicates, then split into train, validation and test
NEAR = 12  # differing bits out of 256 that still count as "near-identical"

def dhash(path, size=16):
    g = Image.open(path).convert('L').resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten())  # 256 bits

n = len(df)
parent = list(range(n))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

hashes = np.stack([dhash(p) for p in df.path]).view(np.uint64)
if hasattr(np, 'bitwise_count'):
    popcount = lambda x: np.bitwise_count(x).sum(axis=1)
else:
    _bits = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
    popcount = lambda x: _bits[x.view(np.uint8)].sum(axis=1)

near_pairs = 0
for i in range(n - 1):
    dist = popcount(np.bitwise_xor(hashes[i], hashes[i + 1:]))
    for j in np.nonzero(dist <= NEAR)[0]:
        union(i, i + 1 + int(j))
        near_pairs += 1

if PATIENT_CSV:
    pm = pd.read_csv(PATIENT_CSV)
    pid_by_file = dict(zip(pm['file'].astype(str), pm['patient_id'].astype(str)))
    first = {}
    for i, src in enumerate(df.source):
        pid = pid_by_file.get(os.path.basename(src))
        if pid is None:
            continue
        if pid in first:
            union(first[pid], i)
        else:
            first[pid] = i

df['group'] = [find(i) for i in range(n)]
print(f'{plural(near_pairs, "near-identical pair")} linked; {df.group.nunique()} independent groups from {n} images.')
biggest = int(df.group.value_counts().iloc[0])
if biggest > max(20, 0.05 * n):
    print(f'Note: one linked group holds {biggest} images. Check for repeated video frames in your folders.')
mixed = int((df.groupby('group').label.nunique() > 1).sum())
if mixed:
    print(f'Note: {mixed} linked groups contain more than one condition. Check those labels.')

y, g = df.label.values, df.group.values
outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
trval_idx, test_idx = next(outer.split(np.zeros(n), y, g))
inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
tr_sub, va_sub = next(inner.split(np.zeros(len(trval_idx)), y[trval_idx], g[trval_idx]))
df['split'] = 'test'
df.loc[trval_idx[tr_sub], 'split'] = 'train'
df.loc[trval_idx[va_sub], 'split'] = 'val'
df.to_csv(os.path.join(OUT_DIR, 'split.csv'), index=False)
print(pd.crosstab(df.label, df.split).reindex(columns=['train', 'val', 'test'], fill_value=0))

## Train
The model starts from ImageNet weights. It first trains only its new final layer, then fine-tunes the whole network, and keeps the version that scored best on the validation images. Colour is diagnostic in otoscopy, so the training never shifts hue.

In [ ]:
#@title 5. Prepare the data and model
CLASSES = sorted(df.label.unique())
IDX = {c: i for i, c in enumerate(CLASSES)}
SIZE = {'efficientnet_b0': 224, 'efficientnet_b2': 260, 'convnext_tiny': 224}[ARCH]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(SIZE, scale=(0.7, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(),  # right and left ears are mirror images
    T.RandomRotation(20),      # the speculum view can be rotated
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),  # hue untouched
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((SIZE, SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class EarImages(Dataset):
    def __init__(self, frame, tf):
        self.paths = frame.path.tolist()
        self.ys = [IDX[label] for label in frame.label]
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB')), self.ys[i]

def make_loader(split, train=False):
    part = df[df.split == split]
    return DataLoader(EarImages(part, train_tf if train else eval_tf), batch_size=BATCH,
                      shuffle=train, num_workers=2, pin_memory=(DEVICE == 'cuda'))

train_dl, val_dl, test_dl = make_loader('train', True), make_loader('val'), make_loader('test')

def build_model(n_classes):
    W = torchvision.models
    if ARCH == 'efficientnet_b0':
        m = W.efficientnet_b0(weights=W.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    elif ARCH == 'efficientnet_b2':
        m = W.efficientnet_b2(weights=W.EfficientNet_B2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    elif ARCH == 'convnext_tiny':
        m = W.convnext_tiny(weights=W.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_classes)
    else:
        raise ValueError('Unknown ARCH: ' + ARCH)
    return m

train_counts = df[df.split == 'train'].label.value_counts().reindex(CLASSES).fillna(0).values
weights = torch.tensor(train_counts.sum() / (len(CLASSES) * np.maximum(train_counts, 1)), dtype=torch.float32)
print('Conditions:', ', '.join(CLASSES))
print('Images (train / validation / test):', len(train_dl.dataset), '/', len(val_dl.dataset), '/', len(test_dl.dataset))

In [ ]:
#@title 6. Train (roughly 15–60 minutes for a few thousand images on a T4)
def run_epoch(model, dl, loss_fn, opt=None, scaler=None):
    training = opt is not None
    model.train(training)
    logits_all, y_all, total = [], [], 0.0
    for x, y in dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            out = model(x)
            loss = loss_fn(out, y)
        if training:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        total += loss.item() * len(y)
        logits_all.append(out.float().detach().cpu())
        y_all.append(y.cpu())
    return total / len(dl.dataset), torch.cat(logits_all).numpy(), torch.cat(y_all).numpy()

model = build_model(len(CLASSES)).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=weights.to(DEVICE), label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

# Stage 1: train only the new final layer
for p in model.features.parameters():
    p.requires_grad = False
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
for epoch in range(3):
    tl, _, _ = run_epoch(model, train_dl, loss_fn, opt, scaler)
    vl, vlog, vy = run_epoch(model, val_dl, loss_fn)
    print(f'warm-up {epoch + 1}/3  train loss {tl:.3f}  val loss {vl:.3f}  val acc {(vlog.argmax(1) == vy).mean():.1%}')

# Stage 2: fine-tune the whole network
for p in model.parameters():
    p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
best, best_state, waited = -1.0, None, 0
for epoch in range(EPOCHS):
    tl, _, _ = run_epoch(model, train_dl, loss_fn, opt, scaler)
    vl, vlog, vy = run_epoch(model, val_dl, loss_fn)
    sched.step()
    vbal = balanced_accuracy_score(vy, vlog.argmax(1))
    print(f'epoch {epoch + 1:2d}/{EPOCHS}  train loss {tl:.3f}  val loss {vl:.3f}  '
          f'val acc {(vlog.argmax(1) == vy).mean():.1%}  balanced {vbal:.1%}')
    if vbal > best:
        best, waited = vbal, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        waited += 1
        if waited >= 7:
            print('No improvement for 7 epochs, so training stopped early.')
            break
model.load_state_dict(best_state)
torch.save(best_state, os.path.join(OUT_DIR, f'auris_{ARCH}.pt'))
print(f'Best validation balanced accuracy: {best:.1%}')

## Test honestly
The test images were never used for training or for choosing settings. The **likely range** is a 95% confidence interval: with few test images it is wide.

**Confident mode:** Auris gives a firm answer only when its probability is above a threshold, and flags the rest for specialist review. The threshold is chosen on the validation images, then checked once on the test images. The 95% target counts as met only when the lower end of the likely range is 95% or more, on at least 30 confident test cases.

In [ ]:
#@title 7. Test on images the model has never seen
def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def fit_temperature(logits, labels):
    """Calibrates the probabilities on validation images (temperature scaling)."""
    t = torch.ones(1, requires_grad=True)
    lg, lb = torch.tensor(logits), torch.tensor(labels)
    opt_t = torch.optim.LBFGS([t], lr=0.05, max_iter=200)
    ce = nn.CrossEntropyLoss()
    def closure():
        opt_t.zero_grad()
        loss = ce(lg / t.clamp(min=0.05), lb)
        loss.backward()
        return loss
    opt_t.step(closure)
    return float(t.clamp(min=0.05).item())

_, val_logits, val_y = run_epoch(model, val_dl, loss_fn)
_, test_logits, test_y = run_epoch(model, test_dl, loss_fn)
TEMP = fit_temperature(val_logits, val_y)
val_p, test_p = softmax(val_logits / TEMP), softmax(test_logits / TEMP)

pred = test_p.argmax(1)
correct = pred == test_y
n_test, k_test = len(test_y), int(correct.sum())
lo, hi = wilson(k_test, n_test)
bal = float(balanced_accuracy_score(test_y, pred))
print(f'Test accuracy: {k_test}/{n_test} = {k_test / n_test:.1%}  (likely range {lo:.1%} to {hi:.1%})')
print(f'Balanced accuracy (every condition counts equally): {bal:.1%}\n')

rows = []
for i, c in enumerate(CLASSES):
    tp = int(((pred == i) & (test_y == i)).sum()); fn = int(((pred != i) & (test_y == i)).sum())
    fp = int(((pred == i) & (test_y != i)).sum()); tn = int(((pred != i) & (test_y != i)).sum())
    rows.append({'condition': c, 'test images': tp + fn,
                 'sensitivity': tp / max(tp + fn, 1), 'specificity': tn / max(tn + fp, 1),
                 'precision': tp / max(tp + fp, 1)})
per_class = pd.DataFrame(rows)
display(per_class.style.format({'sensitivity': '{:.0%}', 'specificity': '{:.0%}', 'precision': '{:.0%}'}))

cm = confusion_matrix(test_y, pred, labels=list(range(len(CLASSES))))
fig, ax = plt.subplots(figsize=(1.5 + 0.7 * len(CLASSES), 1.2 + 0.6 * len(CLASSES)))
ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES)), CLASSES)
for (r, c_), v in np.ndenumerate(cm):
    ax.text(c_, r, int(v), ha='center', va='center', color='white' if v > cm.max() * 0.6 else 'black')
ax.set_xlabel('Model said'); ax.set_ylabel('ENT label')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=150); plt.show()

def sweep(p, y):
    conf, ok = p.max(1), p.argmax(1) == y
    out = []
    for t in np.round(np.arange(0.30, 0.995, 0.01), 2):
        m = conf >= t
        nn_, kk = int(m.sum()), int(ok[m].sum())
        out.append({'threshold': float(t), 'n': nn_, 'coverage': nn_ / len(y),
                    'accuracy': kk / nn_ if nn_ else float('nan'), 'ci_low': wilson(kk, nn_)[0]})
    return pd.DataFrame(out)

vs = sweep(val_p, val_y)
ok_rows = vs[(vs.ci_low >= TARGET) & (vs.n >= 30)]
if len(ok_rows):
    THRESH = float(ok_rows.threshold.iloc[0])
else:
    THRESH = 0.90
    print('No threshold reached 95% on the validation images with enough cases, so 0.90 is used for now.')

m = test_p.max(1) >= THRESH
cn, ck = int(m.sum()), int(correct[m].sum())
clo, chi = wilson(ck, cn)
coverage = cn / n_test
target_met = bool(cn >= 30 and clo >= TARGET)
print(f'\nConfident mode (probability ≥ {THRESH:.2f}): {ck}/{cn} correct = {ck / max(cn, 1):.1%} '
      f'(likely range {clo:.1%} to {chi:.1%}), on {coverage:.0%} of test images.')
if target_met:
    print('✅ TARGET MET: in confident mode the whole likely range is at or above 95%.')
else:
    weak = per_class[(per_class.sensitivity < 0.9) | (per_class['test images'] < 20)].condition.tolist()
    print('❌ TARGET NOT MET yet. Add more ENT-confirmed images, especially for:', ', '.join(weak) or 'all conditions')
per_class.to_csv(os.path.join(OUT_DIR, 'test_report.csv'), index=False)

## Export for the Android app
This saves one self-contained `auris.onnx` file. It takes RGB values from 0 to 1 and does the normalisation and calibration inside, so the app only has to resize the photo. The cell checks that the ONNX file gives the same answers as the trained PyTorch model before saving anything for the app.

In [ ]:
#@title 8. Export the model files
import onnx
import onnxruntime as ort

class Deployable(nn.Module):
    """RGB in 0..1 (NCHW) in, calibrated probabilities out."""
    def __init__(self, net, temperature):
        super().__init__()
        self.net = net
        self.register_buffer('mean', torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(STD).view(1, 3, 1, 1))
        self.temperature = float(temperature)
    def forward(self, x):
        return torch.softmax(self.net((x - self.mean) / self.std) / self.temperature, dim=1)

deploy = Deployable(model.float().cpu().eval(), TEMP).eval()
dummy = torch.rand(1, 3, SIZE, SIZE)
onnx_path = os.path.join(OUT_DIR, 'auris.onnx')
tmp_path = '/content/auris_tmp.onnx'

attempts = [
    ('new exporter', dict(dynamo=True, external_data=False)),
    ('new exporter', dict(dynamo=True)),
    ('classic exporter', dict(dynamo=False, opset_version=17)),
]
for label, kwargs in attempts:
    try:
        torch.onnx.export(deploy, (dummy,), tmp_path, input_names=['image'], output_names=['probs'], **kwargs)
        print('Exported with the', label)
        break
    except Exception as e:
        print(f'The {label} failed: {type(e).__name__}: {str(e)[:200]}')
else:
    raise RuntimeError('ONNX export failed. Update PyTorch in Colab and run this cell again.')

# Keep the weights inside one file so the app can load it from its assets
onnx.save_model(onnx.load(tmp_path), onnx_path, save_as_external_data=False)
onnx.checker.check_model(onnx_path)

# The ONNX file must give the same answers as PyTorch
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
in_name = sess.get_inputs()[0].name
raw_tf = T.Compose([T.Resize((SIZE, SIZE)), T.ToTensor()])
worst = 0.0
for p in df[df.split == 'test'].path.head(10):
    x = raw_tf(Image.open(p).convert('RGB')).unsqueeze(0)
    with torch.no_grad():
        ref = deploy(x).numpy()
    out = sess.run(None, {in_name: x.numpy()})[0]
    worst = max(worst, float(np.abs(ref - out).max()))
print(f'Largest difference between PyTorch and ONNX: {worst:.2e}')
assert worst < 1e-3, 'The ONNX output differs from PyTorch, so this file should not be used.'

with open(os.path.join(OUT_DIR, 'labels.txt'), 'w') as f:
    f.write('\n'.join(CLASSES) + '\n')

info = {
    'name': f'Auris {ARCH}',
    'arch': ARCH,
    'trained_on': datetime.date.today().isoformat(),
    'input_size': SIZE,
    'input': 'float32 [1, 3, H, W], RGB scaled to 0..1',
    'labels': CLASSES,
    'temperature': TEMP,
    'train_images': int((df.split == 'train').sum()),
    'val_images': int((df.split == 'val').sum()),
    'test_images': n_test,
    'test_accuracy': k_test / n_test,
    'test_ci_low': lo,
    'test_ci_high': hi,
    'balanced_accuracy': bal,
    'threshold': THRESH,
    'confident_accuracy': ck / cn if cn else None,
    'confident_ci_low': clo if cn else None,
    'confident_coverage': coverage,
    'target': TARGET,
    'target_met': target_met,
    'per_class': per_class.to_dict(orient='records'),
}
with open(os.path.join(OUT_DIR, 'model_info.json'), 'w') as f:
    json.dump(info, f, indent=2)

zip_path = '/content/auris_model.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    for fn in ['auris.onnx', 'labels.txt', 'model_info.json', 'test_report.csv', 'confusion_matrix.png', 'split.csv']:
        fp = os.path.join(OUT_DIR, fn)
        if os.path.exists(fp):
            z.write(fp, fn)
print(f'Saved to {OUT_DIR} and {zip_path}. Model size: {os.path.getsize(onnx_path) / 1e6:.1f} MB')

from google.colab import files
files.download(zip_path)

## Next: put the model in the app
1. Unzip `auris_model.zip`.
2. Copy `auris.onnx`, `labels.txt` and `model_info.json` into the Android project folder `app/src/main/assets/`.
3. Rebuild the app. Its Accuracy tab shows this test report.

**Reading the result honestly**
- The test images come from the same sources as the training images. Other clinics use other otoscopes, lighting and patients, so expect lower accuracy there until you have measured it.
- Before doctors rely on Auris, test it on images from a clinic that supplied none of the training data, and record verdicts in the app.
- Rare conditions need the most new images. The per-condition table shows where to focus.